# 06 — Spectral Velocity Cycle Analysis

**This is where your formula really lives.**

Standard momentum asks: *is this moving?*

Spectral velocity asks: *is this moving faster or slower than it should be,
given where we are in the cycle?*

The `velocity_surprise` = `current_velocity` − `expected_velocity_for_this_phase`

This is directly analogous to how the ag system detects crop yield surprises —
not whether NDVI is high, but whether it's rising **faster than the seasonal norm**.

---

### Cycle phases (mirroring crop phenology)

| Financial | Crop analog | Description |
|-----------|------------|-------------|
| 🟢 green_up | Green-up | Velocity rising, below peak. Early expansion. |
| 🟡 peak | Peak NDVI | Velocity at maximum. Late expansion. |
| 🔴 senescence | Senescence | Velocity falling from peak. Contraction. |
| ⚫ dormant | Dormancy | Velocity at minimum. Trough / base. |

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..', 'specvel'))
os.makedirs('../results', exist_ok=True)

import pandas as pd
import matplotlib.pyplot as plt

from cycle       import compute_velocity_surprise, run_cycle_scan, print_cycle_leaderboard
from cycle_chart import plot_cycle_analysis, plot_cycle_dashboard

FRED_KEY = os.environ.get('FRED_API_KEY', 'your_key_here')

START_EQ  = '2020-01-01'
START_COM = '2020-01-01'
START_FI  = '2018-01-01'
START_MAC = '2010-01-01'
END       = '2026-03-10'

## 1. Single Series Deep Dive — S&P 500

Four-panel chart: phase-colored price, velocity + expected band, surprise bars, phase timeline.

In [ ]:
from adapters.equities import EquitiesAdapter

adapter = EquitiesAdapter()
raw     = adapter.fetch('SPY', START_EQ, END)
normed  = adapter.normalize(raw)

# Equities use business cycle method
surp = compute_velocity_surprise(normed, cycle_method='business')

print(f"Current phase:      {surp['current_phase'].upper()}")
print(f"Phase age:          {surp['phase_age']} periods")
print(f"Current velocity:   {surp['current_velocity']:+.6f}")
print(f"Expected velocity:  {surp['expected_velocity']:+.6f}")
print(f"Velocity surprise:  {surp['velocity_surprise']:+.6f}")
print(f"Surprise z-score:   {surp['surprise_zscore']:+.4f}")
print(f"Surprise signal:    {surp['surprise_signal']}")
print(f"Conviction boost:   {surp['conviction_boost']:+d}")
if surp['transition_warning']:
    print(f"⚠  Warning:         {surp['transition_warning']}")

In [ ]:
fig = plot_cycle_analysis(
    normed, surp,
    title='S&P 500 (SPY)',
    save_path='../results/spy_cycle_analysis.png'
)

## 2. Equities Cycle Scan — Full Universe

In [ ]:
df_eq = run_cycle_scan(
    adapter, start=START_EQ, end=END,
    cycle_method='business', top_n=20
)
print_cycle_leaderboard(df_eq, title='EQUITIES CYCLE SCAN')

In [ ]:
fig = plot_cycle_dashboard(
    df_eq, adapter, START_EQ, END,
    top_n=6,
    save_path='../results/equities_cycle_dashboard.png'
)

## 3. Commodities — Auto Cycle Detection

In [ ]:
from adapters.commodities import CommoditiesAdapter

com_adapter = CommoditiesAdapter()
df_com = run_cycle_scan(
    com_adapter, start=START_COM, end=END,
    cycle_method='auto', top_n=15
)
print_cycle_leaderboard(df_com, title='COMMODITIES CYCLE SCAN')

In [ ]:
# Deep dive — WTI Crude
raw    = com_adapter.fetch('CL=F', START_COM, END)
normed = com_adapter.normalize(raw)
surp   = compute_velocity_surprise(normed, cycle_method='auto')

print(f"WTI Crude — {surp['current_phase'].upper()}  "
      f"z={surp['surprise_zscore']:+.3f}  {surp['surprise_signal']}")

fig = plot_cycle_analysis(
    normed, surp,
    title='WTI Crude Oil',
    save_path='../results/wti_cycle_analysis.png'
)

## 4. Fixed Income — Business Cycle Phases

Requires FRED key.

In [ ]:
if FRED_KEY != 'your_key_here':
    from adapters.fixed_income import FixedIncomeAdapter
    fi_adapter = FixedIncomeAdapter(api_key=FRED_KEY)

    df_fi = run_cycle_scan(
        fi_adapter, start=START_FI, end=END,
        cycle_method='business', top_n=15
    )
    print_cycle_leaderboard(df_fi, title='FIXED INCOME CYCLE SCAN')

    # Deep dive — 10Y Treasury
    raw    = fi_adapter.fetch('DGS10', START_FI, END)
    normed = fi_adapter.normalize(raw)
    surp   = compute_velocity_surprise(normed, cycle_method='business')
    print(f"\n10Y Treasury — {surp['current_phase'].upper()}  "
          f"z={surp['surprise_zscore']:+.3f}  {surp['surprise_signal']}")
    fig = plot_cycle_analysis(
        normed, surp,
        title='10Y Treasury Yield',
        save_path='../results/dgs10_cycle_analysis.png'
    )
else:
    print('Set FRED_KEY to run this section.')

## 5. Macro — Calendar Cycle

CPI and retail sales have strong Q1-Q4 seasonal structure.
Requires FRED key.

In [ ]:
if FRED_KEY != 'your_key_here':
    from adapters.macro import MacroAdapter
    mac_adapter = MacroAdapter(api_key=FRED_KEY)

    df_mac = run_cycle_scan(
        mac_adapter, start=START_MAC, end=END,
        cycle_method='calendar', top_n=15
    )
    print_cycle_leaderboard(df_mac, title='MACRO CYCLE SCAN')

    # Deep dive — CPI
    raw    = mac_adapter.fetch('CPIAUCSL', START_MAC, END)
    normed = mac_adapter.normalize(raw)
    surp   = compute_velocity_surprise(normed, cycle_method='calendar')
    print(f"\nCPI — {surp['current_phase'].upper()}  "
          f"z={surp['surprise_zscore']:+.3f}  {surp['surprise_signal']}")
    fig = plot_cycle_analysis(
        normed, surp,
        title='CPI All Items',
        save_path='../results/cpi_cycle_analysis.png'
    )
else:
    print('Set FRED_KEY to run this section.')

## 6. Cross-Asset Surprise Summary

Which assets have the biggest velocity surprise RIGHT NOW across all asset classes?

In [ ]:
import pandas as pd

frames = [df_eq]
if 'df_com' in dir(): frames.append(df_com)
if 'df_fi'  in dir(): frames.append(df_fi)
if 'df_mac' in dir(): frames.append(df_mac)

combined = pd.concat(frames, ignore_index=True)
combined = combined.sort_values('surprise_zscore', key=abs, ascending=False)

print_cycle_leaderboard(combined.head(20), title='CROSS-ASSET VELOCITY SURPRISE — TOP 20')

In [ ]:
# Save all results
for name, df in [('equities', df_eq), ('commodities', df_com if 'df_com' in dir() else pd.DataFrame())]:
    if not df.empty:
        save_cols = [c for c in df.columns if not c.startswith('_')]
        df[save_cols].to_csv(f'../results/{name}_cycle_scan.csv', index=False)
        print(f'Saved ../results/{name}_cycle_scan.csv')